# 01 · Load & clean the data 🧹

**Goal:** turn the raw OAIC figures into one clean, analysis-ready table saved to `data/processed/`.

**In this notebook you'll:**
1. Load and explore the raw CSV (2018 to Jan–Jun 2025)
2. Run some data quality checks
3. Parse the Jul–Dec 2025 OAIC spreadsheet into the same tidy format
4. Combine everything
5. Convert the 2018–19 quarterly figures into half-years
6. Save the result

> 💡 Each section has a short explanation and **hints** in the code cells. Write the code yourself, run it, and look at the output before moving on. Stuck? Ask Claude for a nudge (not the answer 😉).

## 0. Setup
This cell is done for you. Run it with **Shift+Enter**. If it errors, check VS Code is using the `.venv` kernel (top right of the notebook).

In [ ]:
from pathlib import Path

import pandas as pd

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

RAW_CSV = DATA_RAW / "oaic_ndb_raw.csv"
XLSX_2025H2 = DATA_RAW / "ndb-data-1-july-2025-to-31-dec-2025.xlsx"

pd.set_option("display.max_columns", None)
print("Ready ✨")

## 1. Load & explore the raw CSV

Load `RAW_CSV` into a DataFrame called `raw`. Make `period_start` and `period_end` proper dates rather than text.

Then explore it: how many rows? Which metrics are there, and how many rows each? Which periods?

In [ ]:
# Hint: pd.read_csv(..., parse_dates=[...])
raw = ...


In [ ]:
# Hint: .shape, .head(), .info()


In [ ]:
# Hint: value_counts() on the 'metric' column, then on 'period_start'


## 2. Data quality checks 🔍

Good analysts never trust data blindly. For each period, check that the **breach source counts add up to the total notifications**.

Steps:
1. Keep only rows where `unit == "count"`
2. Get the totals (`metric == "total_notifications"`) per `period_start`
3. Sum `breaches_by_source` per `period_start`
4. Put them side by side and add a `difference` column

**Think about it:** which periods don't match, and why? (Check `data/README.md` → *Coverage & known gaps*.) Jot your answer in the markdown cell below.

In [ ]:
# Hint: filter with raw[raw["unit"] == "count"]
# Hint: groupby("period_start")["value"].sum()
# Hint: pd.concat([totals, source_sums], axis=1)  or  .to_frame().join(...)


**My notes on the mismatches:**

- ...

## 3. Parse the Jul–Dec 2025 spreadsheet 🧩

The OAIC spreadsheet uses a **nested layout**: a heading row (like `July`) is followed by its sub-rows (like `Human error`). Start by looking at the `NDB by month` sheet exactly as it comes in.

In [ ]:
# Hint: pd.read_excel(XLSX_2025H2, sheet_name="NDB by month", header=2)
#       header=2 tells pandas the column names are on the 3rd row
month_sheet = ...


Now reshape it. The trick:

1. Rename the columns to something simple, e.g. `label` and `count`
2. Make a `month` column: where `label` is a month name, copy it; otherwise leave it empty
3. **Forward-fill** the `month` column so every source row knows which month it belongs to
4. Split into **month total rows** and **source rows**
5. Sum the month totals → total notifications for the half-year
6. Sum the source rows by `label` → breaches by source for the half-year

⚠️ **Gotcha:** look at the very last row of the sheet. Is it a month, a source, or something else? Make sure it doesn't sneak into your sums 👀

In [ ]:
MONTHS = ["July", "August", "September", "October", "November", "December"]

# Hint: df["month"] = df["label"].where(df["label"].isin(MONTHS))
# Hint: df["month"] = df["month"].ffill()
# Hint: you may need to drop empty / note rows first with .dropna()


In [ ]:
# Hint: is_month_row = df["label"].isin(MONTHS)
# Hint: total_2025h2 = df.loc[is_month_row, "count"].sum()
# Hint: source_2025h2 = df.loc[~is_month_row].groupby("label")["count"].sum()


**Check:** your half-year total should be **670**, matching the *Grand Total* on the `Individuals affected` sheet. Does it?

Now turn your results into rows with the **same columns as `raw`**, so they can be combined.

Use the same category names as the CSV: `malicious_or_criminal_attack`, `human_error`, `system_fault`. The new sources `Other` and `Currently unknown` can become `other` and `currently_unknown`.

In [ ]:
SOURCE_NAMES = {
    "Malicious or criminal attack": "malicious_or_criminal_attack",
    "Human error": "human_error",
    "System fault": "system_fault",
    "Other": "other",
    "Currently unknown": "currently_unknown",
}

# Hint: build a list of dicts, one per row, then pd.DataFrame(list_of_dicts)
# Each dict needs: period_start, period_end, frequency, metric, category,
#                  subcategory, value, unit, source_report, page, notes
new_2025h2 = ...


## 4. Combine everything
Stack `raw` and `new_2025h2` into one DataFrame called `combined`, then check the row count went up by the right amount.

In [ ]:
# Hint: pd.concat([raw, new_2025h2], ignore_index=True)
combined = ...


## 5. Quarterly → half-yearly 📅

2018 to mid-2019 were reported **quarterly**; everything after is **half-yearly**. To compare fairly, add a `half_year` label to every row, e.g. `2018-H1`, `2018-H2`.

- `half_year` = the year + `H1` if the period starts in Jan–Jun, otherwise `H2`
- Then, for **count** rows only, sum `value` by `half_year`, `metric`, `category` and `subcategory`

**Think about it ⚠️:** why can't you add up the **percent** rows the same way? What does that mean for Jan–Jun 2018?

In [ ]:
# Hint: combined["period_start"].dt.year and .dt.month
# Hint: np.where(month <= 6, "H1", "H2")  (import numpy as np)  or  .map / apply


In [ ]:
# Hint: counts = combined[combined["unit"] == "count"]
# Hint: counts.groupby(["half_year", "metric", "category", "subcategory"], dropna=False)["value"].sum().reset_index()
half_yearly = ...


**My notes on percent rows:**

- ...

## 6. Save 💾
Save `half_yearly` to `DATA_PROCESSED / "ndb_half_yearly.csv"` (no index), then commit and push your notebook and the new file.

```bash
git add notebooks/ data/processed/
git commit -m "Load and clean OAIC data into half-yearly table"
git push
```

In [ ]:
# Hint: half_yearly.to_csv(..., index=False)


## ✅ Wrap-up

In 2–3 bullet points, what did you learn about the data's quality and limitations? These go straight into your README later.

- ...